In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_table("total_tads_histonesignal.txt",header=None)
df.columns = ["chr", "start", "end", "wt-k27ac_1", "wt-k27ac_2",'wt-4me3_1','wt-4me3_2','wt-27me3_1','wt-27me3_2', "wt-input_1", "wt-input_2", "pi-k27ac_1", "pi-k27ac_2",'pi-4me3_1','pi-4me3_2','pi-27me3_1','pi-27me3_2',"pi-input_1", "pi-input_2","TADs"]
df.head()

In [ ]:
#calibrate depth
df["wt-k27ac_1nor"] = df["wt-k27ac_1"]/47
df["wt-k27ac_2nor"] = df["wt-k27ac_2"]/53
df["wt-4me3_1nor"] = df["wt-4me3_1"]/37
df["wt-4me3_2nor"] = df["wt-4me3_2"]/34
df["wt-k27me3_1nor"] = df["wt-27me3_1"]/63
df["wt-k27me3_2nor"] = df["wt-27me3_2"]/73
df["wt-igg_1nor"] = df["wt-input_1"]/10
df["wt-igg_2nor"] = df["wt-input_2"]/10
df["pi-k27ac_1nor"] = df["pi-k27ac_1"]/38
df["pi-k27ac_2nor"] = df["pi-k27ac_2"]/59
df["pi-4me3_1nor"] = df["pi-4me3_1"]/13
df["pi-4me3_2nor"] = df["pi-4me3_2"]/58
df["pi-k27me3_1nor"] = df["pi-27me3_1"]/26
df["pi-k27me3_2nor"] = df["pi-27me3_2"]/46
df["pi-igg_1nor"] = df["pi-input_1"]/8
df["pi-igg_2nor"] = df["pi-input_2"]/9
df.head()

In [ ]:
df['wt-k27ac_mean'] = df[['wt-k27ac_1nor', 'wt-k27ac_2nor']].mean(axis=1)
df['wt-4me3_mean'] = df[['wt-4me3_1nor', 'wt-4me3_2nor']].mean(axis=1)
df['wt-k27me3_mean'] = df[['wt-k27me3_1nor', 'wt-k27me3_2nor']].mean(axis=1)
df['wt-igg_mean'] = df[['wt-igg_1nor', 'wt-igg_2nor']].mean(axis=1)

In [ ]:
df['pi-k27ac_mean'] = df[['pi-k27ac_1nor', 'pi-k27ac_2nor']].mean(axis=1)
df['pi-4me3_mean'] = df[['pi-4me3_1nor', 'pi-4me3_2nor']].mean(axis=1)
df['pi-k27me3_mean'] = df[['pi-k27me3_1nor', 'pi-k27me3_2nor']].mean(axis=1)
df['pi-igg_mean'] = df[['pi-igg_1nor', 'pi-igg_2nor']].mean(axis=1)

In [ ]:
df.head()

In [ ]:
df["wt-k27ac_change"] = np.log2((df["wt-k27ac_mean"] + 1e-3)/(df["wt-igg_mean"] + 1e-3))
df["wt-4me3_change"] = np.log2((df["wt-4me3_mean"] + 1e-3)/(df["wt-igg_mean"] + 1e-3))
df["wt-k27me3_change"] = np.log2((df["wt-k27me3_mean"] + 1e-3)/(df["wt-igg_mean"] + 1e-3))
df["pi-k27ac_change"] = np.log2((df["pi-k27ac_mean"] + 1e-3)/(df["pi-igg_mean"] + 1e-3))
df["pi-4me3_change"] = np.log2((df["pi-4me3_mean"] + 1e-3)/(df["pi-igg_mean"] + 1e-3))
df["pi-k27me3_change"] = np.log2((df["pi-k27me3_mean"] + 1e-3)/(df["pi-igg_mean"] + 1e-3))
df.head()

In [ ]:
df.loc[df['wt-k27ac_change'] < 0, 'wt-k27ac_change'] = 0
df.loc[df['wt-4me3_change'] < 0, 'wt-4me3_change'] = 0
df.loc[df['wt-k27me3_change'] < 0, 'wt-k27me3_change'] = 0
df.loc[df['pi-k27ac_change'] < 0, 'pi-k27ac_change'] = 0
df.loc[df['pi-4me3_change'] < 0, 'pi-4me3_change'] = 0
df.loc[df['pi-k27me3_change'] < 0, 'pi-k27me3_change'] = 0
df.head()

In [ ]:
h3k27ac = df[["chr","start","end","wt-k27ac_change","pi-k27ac_change","TADs"]] 
h3k27ac.head()
h3k27ac.to_csv("h3k27ac_change.txt", sep="\t", index=False)

In [ ]:
h3k27me3 = df[["chr","start","end","wt-k27me3_change","pi-k27me3_change","TADs"]] 
h3k27me3.head()
h3k27me3.to_csv("h3k27me3_change.txt", sep="\t", index=False)

In [ ]:
h3k4me3 = df[["chr","start","end","wt-4me3_change","pi-4me3_change","TADs"]] 
h3k4me3.head()
h3k4me3.to_csv("h3k4me3_change.txt", sep="\t", index=False)

In [ ]:
df_filtered = h3k4me3[~((h3k4me3["wt-4me3_change"] == 0) & (h3k4me3["pi-4me3_change"] == 0))]

In [ ]:
df_long = df_filtered.melt(
    id_vars=["chr", "start", "end", "TADs"],
    value_vars=["wt-4me3_change", "pi-4me3_change"],
    var_name="Group",
    value_name="Signal"
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    x="TADs",            # Group by TADs
    y="Signal",          # Y-axis is signal
    hue="Group",         # Distinguish wt and pi
    data=df_long,
    palette="Set2",
    showfliers=False,    # Remove outliers
    width=0.6
)

# 5. Beautify the plot
plt.title("K27ac Change by TADs", fontsize=14)
plt.xlabel("TADs", fontsize=12)
plt.ylabel("K27ac Change", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.3)
sns.despine()
plt.legend(title="Group", loc="upper right")

# 6. Save the figure
plt.tight_layout()
#plt.savefig("k27ac_change_by_TADs_boxplot.png", dpi=300)
plt.show()

In [ ]:
df = pd.read_table("h3k4me3_change.txt")
df.head()

In [ ]:
df['change'] = df['pi-4me3_change']-df['wt-4me3_change']
df.head()

In [ ]:
df_filtered = df[~((df["wt-4me3_change"] == 0) & (df["pi-4me3_change"] == 0))]

In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator
from scipy.stats import mannwhitneyu, normaltest

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax = sns.boxplot(x='TADs', y='change', hue="TADs", data=df_filtered, showfliers=False,width=0.5)
plt.title("H3K4me3 FC of TAD")
plt.savefig("tad_4me3_change.pdf")
plt.show()